In [1]:
import os
import requests
import pandas as pd

# Ensure data/raw directory exists from project root
os.makedirs("data/raw", exist_ok=True)

# 1. Fetch KNBS CPI data
url = "https://api.worldbank.org/v2/country/KEN/indicator/FP.CPI.TOTL?date=2015:2026&format=json"
res = requests.get(url).json()
records = [{"date": f"{e['date']}-12-01", "cpi_index": e['value']} for e in res[1] if e['value'] is not None]
pd.DataFrame(records).sort_values("date").to_csv("data/raw/knbs_cpi_monthly.csv", index=False)

# 2. Fetch CBK macro indicators
cbk_url = "https://api.worldbank.org/v2/country/KEN/indicator/PA.NUS.FCRF?date=2015:2026&format=json"
res_cbk = requests.get(cbk_url).json()
cbk_records = [{"date": f"{e['date']}-12-01", "central_bank_rate": 9.5, "exchange_rate_usd_kes": e['value']} for e in res_cbk[1] if e['value'] is not None]
pd.DataFrame(cbk_records).sort_values("date").to_csv("data/raw/cbk_macro_indicators.csv", index=False)

print("✅ Successfully re-populated data/raw/ with both CSV files!")

✅ Successfully re-populated data/raw/ with both CSV files!


In [2]:
import pandas as pd
import numpy as np
import os

# Ensure raw data directory exists
os.makedirs("../data/raw", exist_ok=True)

# Set random seed for reproducibility
np.random.seed(42)

# Generate monthly date range from Jan 2015 to Dec 2025 (132 months)
dates = pd.date_range(start="2015-01-01", end="2025-12-01", freq="MS")
n = len(dates)

# Generate baseline trends
cpi = np.linspace(150, 268, n) + np.sin(np.linspace(0, 22*np.pi, n)) * 2 + np.random.normal(0, 0.5, n)
exchange_rate = np.linspace(100, 135, n) + np.cumsum(np.random.normal(0, 0.3, n))
maize_price_kg = 35 + (cpi - 150) * 0.4 + np.sin(np.linspace(0, 11*np.pi, n)) * 6 + np.random.normal(0, 1.5, n)
rainfall_mm = 80 + np.sin(np.linspace(0, 22*np.pi, n) - 1) * 50 + np.random.normal(0, 10, n)

# --- INJECT REAL-WORLD MESSINESS ---
# 1. Random missing values (NaNs) across columns
cpi[np.random.choice(n, 8, replace=False)] = np.nan
exchange_rate[np.random.choice(n, 6, replace=False)] = np.nan
maize_price_kg[np.random.choice(n, 10, replace=False)] = np.nan
rainfall_mm[np.random.choice(n, 7, replace=False)] = np.nan

# 2. Extreme Outliers & Data Entry Errors
maize_price_kg[45] = 9999.0  # Typo / data entry glitch
exchange_rate[102] = 5.5     # Flash crash / reporting error
rainfall_mm[12] = -450.0     # Invalid negative sensor reading

# Construct the raw dataframe
messy_df = pd.DataFrame({
    "date": dates.strftime("%Y-%m-%d"),
    "cpi_index": np.round(cpi, 2),
    "exchange_rate_usd_kes": np.round(exchange_rate, 2),
    "maize_price_kg_kes": np.round(maize_price_kg, 2),
    "rainfall_index_mm": np.round(rainfall_mm, 2)
})

# Save as our raw dataset
messy_df.to_csv("../data/raw/kenya_macro_maize_monthly_raw.csv", index=False)

print("=== RAW DATA AUDIT: MESSY DATASET LOADED ===")
print(f"Total Observations: {len(messy_df)}")
print("\nMissing Values per Feature:")
print(messy_df.isnull().sum())
print("\nStatistical Summary (Notice max/min distortion from outliers):")
display(messy_df.describe())
print("\nSample of Raw Messy Data:")
display(messy_df.head(15))

=== RAW DATA AUDIT: MESSY DATASET LOADED ===
Total Observations: 132

Missing Values per Feature:
date                      0
cpi_index                 8
exchange_rate_usd_kes     6
maize_price_kg_kes       10
rainfall_index_mm         7
dtype: int64

Statistical Summary (Notice max/min distortion from outliers):


,cpi_index,exchange_rate_usd_kes,maize_price_kg_kes,rainfall_index_mm
count,124.000000,126.000000,122.000000,125.000000
mean,208.845000,118.197302,140.981148,74.623040
std,34.737582,15.281736,899.996593,59.415638
min,150.250000,5.500000,33.520000,-450.000000
25%,177.150000,108.817500,47.137500,45.860000
50%,209.455000,118.175000,59.845000,80.340000
75%,240.627500,128.712500,71.750000,108.850000
max,268.030000,137.850000,9999.000000,148.440000



Sample of Raw Messy Data:


,date,cpi_index,exchange_rate_usd_kes,maize_price_kg_kes,rainfall_index_mm
0,2015-01-01,150.25,99.68,34.72,20.80
1,2015-02-01,151.84,100.09,35.43,70.79
2,2015-03-01,153.87,100.08,NaN,81.61
3,2015-04-01,155.46,NaN,39.31,119.90
4,2015-05-01,155.20,100.85,41.64,108.85
5,2015-06-01,155.35,101.02,43.15,NaN
6,2015-07-01,156.15,101.53,45.62,121.47
7,2015-08-01,155.64,101.43,40.88,102.15
8,2015-09-01,155.21,101.76,43.98,71.54
9,2015-10-01,156.38,NaN,41.73,57.72


In [3]:
import pandas as pd
import numpy as np
import os

os.makedirs("../data/raw", exist_ok=True)
np.random.seed(42)

# Create a broad panel dataset across multiple counties and months (2015-2025)
counties = ["Nairobi", "Nakuru", "Uasin Gishu (Eldoret)", "Kisumu", "Mombasa"]
dates = pd.date_range(start="2015-01-01", end="2025-12-01", freq="MS")

records = []
for county in counties:
    for d in dates:
        # Base realistic trends with seasonal volatility
        cpi = 150 + (d.year - 2015) * 10 + np.random.normal(0, 1.2)
        exc = 100 + (d.year - 2015) * 3 + np.random.normal(0, 0.5)
        maize = 35 + (cpi - 150) * 0.45 + np.sin(d.month / 12 * 2 * np.pi) * 8 + np.random.normal(0, 2.5)
        rain = 90 + np.sin((d.month - 3) / 12 * 2 * np.pi) * 60 + np.random.normal(0, 15)
        
        records.append({
            "date": d.strftime("%Y-%m-%d"),
            "county": county,
            "cpi_index": round(cpi, 2),
            "exchange_rate_usd_kes": round(exc, 2),
            "maize_retail_price_kg": round(maize, 2),
            "rainfall_mm": round(rain, 2)
        })

df_raw = pd.DataFrame(records)

# --- INTRODUCE HEAVY REAL-WORLD MESSINESS ---
# 1. Random missing values (NaNs) across columns (~5% corruption)
for col in ["cpi_index", "exchange_rate_usd_kes", "maize_retail_price_kg", "rainfall_mm"]:
    mask = np.random.choice([True, False], size=len(df_raw), p=[0.05, 0.95])
    df_raw.loc[mask, col] = np.nan

# 2. Extreme Outliers and Data Entry Typos
df_raw.loc[12, "maize_retail_price_kg"] = 88888.0   # Impossible typo
df_raw.loc[154, "exchange_rate_usd_kes"] = -12.5  # Invalid negative exchange rate
df_raw.loc[302, "rainfall_mm"] = 9999.0          # Sensor glitch error
df_raw.loc[410, "cpi_index"] = "N/A"             # String contamination in numeric column

# Save as our master raw messy dataset
df_raw.to_csv("../data/raw/kenya_macro_maize_master_messy.csv", index=False)

print("=== MASTER MESSY DATASET CREATED ===")
print(f"Total Rows: {len(df_raw)}")
print(f"Total Columns: {len(df_raw.columns)}")
print("\nMissing Values Breakdown:")
print(df_raw.isnull().sum())
print("\nSample of Raw Messy Data:")
display(df_raw.sample(10))

TypeError: Invalid value 'N/A' for dtype 'float64'

In [4]:
import pandas as pd
import numpy as np
import os

os.makedirs("../data/raw", exist_ok=True)
np.random.seed(42)

counties = ["Nairobi", "Nakuru", "Uasin Gishu (Eldoret)", "Kisumu", "Mombasa"]
dates = pd.date_range(start="2015-01-01", end="2025-12-01", freq="MS")

records = []
for county in counties:
    for d in dates:
        cpi = 150 + (d.year - 2015) * 10 + np.random.normal(0, 1.2)
        exc = 100 + (d.year - 2015) * 3 + np.random.normal(0, 0.5)
        maize = 35 + (cpi - 150) * 0.45 + np.sin(d.month / 12 * 2 * np.pi) * 8 + np.random.normal(0, 2.5)
        rain = 90 + np.sin((d.month - 3) / 12 * 2 * np.pi) * 60 + np.random.normal(0, 15)
        
        records.append({
            "date": d.strftime("%Y-%m-%d"),
            "county": county,
            "cpi_index": round(cpi, 2),
            "exchange_rate_usd_kes": round(exc, 2),
            "maize_retail_price_kg": round(maize, 2),
            "rainfall_mm": round(rain, 2)
        })

df_raw = pd.DataFrame(records)

# --- INTRODUCE HEAVY REAL-WORLD MESSINESS ---
# 1. Random NaNs
for col in ["cpi_index", "exchange_rate_usd_kes", "maize_retail_price_kg", "rainfall_mm"]:
    mask = np.random.choice([True, False], size=len(df_raw), p=[0.05, 0.95])
    df_raw.loc[mask, col] = np.nan

# 2. Extreme Outliers
df_raw.loc[12, "maize_retail_price_kg"] = 88888.0   # Impossible typo
df_raw.loc[154, "exchange_rate_usd_kes"] = -12.5  # Invalid negative exchange rate
df_raw.loc[302, "rainfall_mm"] = 9999.0          # Sensor glitch error

# 3. String contamination (Convert column to object dtype first to allow mixed types like real messy data)
df_raw["cpi_index"] = df_raw["cpi_index"].astype(object)
df_raw.loc[410, "cpi_index"] = "N/A"             # String contamination

# Save as our master raw messy dataset
df_raw.to_csv("../data/raw/kenya_macro_maize_master_messy.csv", index=False)

print("=== MASTER MESSY DATASET CREATED SUCCESSFULLY ===")
print(f"Total Rows: {len(df_raw)}")
print(f"Total Columns: {len(df_raw.columns)}")
print("\nData Types:")
print(df_raw.dtypes)
print("\nSample of Raw Messy Data:")
display(df_raw.sample(10))

=== MASTER MESSY DATASET CREATED SUCCESSFULLY ===
Total Rows: 660
Total Columns: 6

Data Types:
date                         str
county                       str
cpi_index                 object
exchange_rate_usd_kes    float64
maize_retail_price_kg    float64
rainfall_mm              float64
dtype: object

Sample of Raw Messy Data:


,date,county,cpi_index,exchange_rate_usd_kes,maize_retail_price_kg,rainfall_mm
116,2024-09-01,Nairobi,239.87,126.89,68.97,101.36
184,2019-05-01,Nakuru,190.31,112.49,61.30,157.18
123,2025-04-01,Nairobi,250.09,129.57,90.78,128.08
649,2025-02-01,Mombasa,249.27,130.11,89.60,52.62
650,2025-03-01,Mombasa,247.75,130.31,85.40,72.16
183,2019-04-01,Nakuru,190.49,111.15,62.72,127.09
309,2018-10-01,Uasin Gishu (Eldoret),182.01,108.72,43.90,84.43
640,2024-05-01,Mombasa,238.78,127.04,76.64,145.79
642,2024-07-01,Mombasa,241.03,126.88,72.06,120.23
31,2017-08-01,Nairobi,170.7,107.10,34.91,111.51


In [5]:
import os
print("Current Working Directory:", os.getcwd())
print("Files in current directory:", os.listdir('.'))
print("Files in ../data/raw/:", os.listdir('../data/raw/') if os.path.exists('../data/raw/') else "Folder not found here")
print("Files in data/raw/:", os.listdir('data/raw/') if os.path.exists('data/raw/') else "Folder not found here")

Current Working Directory: c:\Users\USER\OneDrive\New folder\Desktop\KENYA_ECONOMIC_INTELLIGENCE_AND_FORECASTING_ENGINE\notebooks
Files in current directory: ['01_data_audit.ipynb', 'data']
Files in ../data/raw/: ['kenya_macro_maize_master_messy.csv', 'kenya_macro_maize_monthly_raw.csv']
Files in data/raw/: ['cbk_macro_indicators.csv', 'knbs_cpi_monthly.csv']


In [6]:
import shutil
import os

# 1. If a stray data folder was created inside notebooks, move or clean it
if os.path.exists("data"):
    print("Cleaning up local notebooks/data directory...")
    shutil.rmtree("data")

print("✅ Project directory structure is aligned. Your master dataset is safely stored at ../data/raw/kenya_macro_maize_master_messy.csv")

Cleaning up local notebooks/data directory...
✅ Project directory structure is aligned. Your master dataset is safely stored at ../data/raw/kenya_macro_maize_master_messy.csv


In [7]:
import pandas as pd
import numpy as np
import os

os.makedirs("../data/raw", exist_ok=True)
np.random.seed(42)

dates = pd.date_range(start="2015-01-01", end="2025-12-01", freq="MS")
n = len(dates)

# 1. CBK Macroeconomic Indicators (Exchange Rate & CBR)
cbk_df = pd.DataFrame({
    "date": dates.strftime("%Y-%m-%d"),
    "exchange_rate_usd_kes": np.round(np.linspace(100, 135, n) + np.cumsum(np.random.normal(0, 0.3, n)), 2),
    "central_bank_rate": np.round(np.random.choice([9.5, 10.0, 10.5, 11.0, 12.5, 13.0], size=n), 2)
})
# Inject messiness (NaNs and invalid negative exchange rate)
cbk_df.loc[np.random.choice(n, 5, replace=False), "exchange_rate_usd_kes"] = np.nan
cbk_df.loc[42, "exchange_rate_usd_kes"] = -15.2
cbk_df.to_csv("../data/raw/cbk_macro_indicators.csv", index=False)

# 2. KNBS Consumer Price Index (CPI)
cpi_df = pd.DataFrame({
    "date": dates.strftime("%Y-%m-%d"),
    "cpi_index": np.round(np.linspace(150, 268, n) + np.sin(np.linspace(0, 22*np.pi, n)) * 2, 2)
})
cpi_df.loc[np.random.choice(n, 4, replace=False), "cpi_index"] = np.nan
cpi_df.to_csv("../data/raw/knbs_cpi_monthly.csv", index=False)

# 3. Ministry of Agriculture / Market Maize Retail Prices across Counties
counties = ["Nairobi", "Nakuru", "Uasin Gishu (Eldoret)", "Kisumu", "Mombasa"]
maize_records = []
for county in counties:
    for d in dates:
        price = 35 + (d.year - 2015) * 2 + np.sin(d.month / 12 * 2 * np.pi) * 8 + np.random.normal(0, 3.0)
        maize_records.append({
            "date": d.strftime("%Y-%m-%d"),
            "county": county,
            "maize_retail_price_kg": round(price, 2)
        })

maize_df = pd.DataFrame(maize_records)
# Inject messiness (NaNs and extreme typo)
maize_df.loc[np.random.choice(len(maize_df), 15, replace=False), "maize_retail_price_kg"] = np.nan
maize_df.loc[120, "maize_retail_price_kg"] = 77777.0
maize_df.to_csv("../data/raw/maize_market_prices_raw.csv", index=False)

print("✅ Successfully generated 3 separate raw source files in data/raw/:")
print("  - cbk_macro_indicators.csv")
print("  - knbs_cpi_monthly.csv")
print("  - maize_market_prices_raw.csv")

✅ Successfully generated 3 separate raw source files in data/raw/:
  - cbk_macro_indicators.csv
  - knbs_cpi_monthly.csv
  - maize_market_prices_raw.csv
